# Lesson 5: Generalization

Goes with the Lesson 5 page of the Forkcast course. Splits, overfitting on purpose, early stopping, weight decay and dropout, and the two classic leaks.

Runtime: CPU is enough. The `%%writefile` cells for `train.py` and `model.py` contain the reference solutions of Lessons 3 and 4 with the new parts marked; keep your own versions if you prefer and add the new parts by hand.

## 1. Clone your repository and install the package

In [ ]:
GITHUB_USER = "your-github-username"   # edit this line
REPO = "forkcast-course"

!git clone https://github.com/{GITHUB_USER}/{REPO}.git
%cd {REPO}
!pip install -e .

## 2. Split: `forkcast/split.py`

In [ ]:
%%writefile forkcast/split.py
"""Lesson 5. Splitting data into training, validation and test sets."""

from __future__ import annotations

import torch
from torch import Tensor

Split = tuple[tuple[Tensor, Tensor], tuple[Tensor, Tensor], tuple[Tensor, Tensor]]


def train_val_test_split(X: Tensor, y: Tensor, fractions: tuple[float, float, float] = (0.7, 0.15, 0.15), seed: int = 0) -> Split:
    """Shuffle the rows with ``torch.randperm`` (seeded) and cut them into three parts.

    Returns ((X_train, y_train), (X_val, y_val), (X_test, y_test)). Sizes are
    round(N * fraction) for train and validation; the test set gets the rest, so
    every row lands in exactly one part.
    """
    raise NotImplementedError("Lesson 5, step 2: perm = torch.randperm(N, generator=...); slice perm into three index tensors; return X[idx], y[idx] for each")

In [ ]:
import importlib, torch
from torch import nn
import matplotlib.pyplot as plt
from forkcast.data import make_orders
from forkcast import split as split_module
importlib.reload(split_module)
from forkcast.split import train_val_test_split
from forkcast.preprocess import Normalizer

X, y = make_orders(2000, seed=0, nonlinear=True)
(X_tr, y_tr), (X_val, y_val), (X_te, y_te) = train_val_test_split(X, y, seed=0)
print(len(X_tr), len(X_val), len(X_te))
norm = Normalizer().fit(X_tr)                       # training part only
Xn_tr, Xn_val, Xn_te = norm.transform(X_tr), norm.transform(X_val), norm.transform(X_te)

## 3. `fit_with_validation`: `forkcast/train.py`

Below `load`. One `train_one_epoch`, one `evaluate`, keep the best `state_dict` with `copy.deepcopy`, stop after `patience` epochs without improvement, load the best weights back before returning.

In [ ]:
%%writefile forkcast/train.py
"""Lesson 3. The training toolkit: one epoch, evaluation, fit, save, load (reference solution)."""

from __future__ import annotations

from pathlib import Path

import torch
from torch import Tensor, nn


def train_one_epoch(model: nn.Module, X: Tensor, y: Tensor, optimizer: torch.optim.Optimizer, loss_fn: nn.Module) -> float:
    """One pass over the data. Returns the loss as a Python float."""
    model.train()
    optimizer.zero_grad()                 # 1. clean slate (Lesson 2's zero_() for every parameter)
    loss = loss_fn(model(X), y)           #    forward
    loss.backward()                       # 2. slopes into every parameter's .grad
    optimizer.step()                      # 3. the update rule, applied to every parameter
    return loss.item()


@torch.no_grad()
def evaluate(model: nn.Module, X: Tensor, y: Tensor, loss_fn: nn.Module) -> float:
    """Loss on data the model is not learning from."""
    model.eval()
    return loss_fn(model(X), y).item()


def fit(model: nn.Module, X: Tensor, y: Tensor, epochs: int = 300, lr: float = 0.1,
        optimizer_cls=torch.optim.SGD, loss_fn: nn.Module | None = None) -> list[float]:
    """Build the optimizer, run ``epochs`` epochs, return one loss per epoch."""
    loss_fn = loss_fn or nn.MSELoss()
    optimizer = optimizer_cls(model.parameters(), lr=lr)
    return [train_one_epoch(model, X, y, optimizer, loss_fn) for _ in range(epochs)]


def save(model: nn.Module, path: str | Path) -> None:
    torch.save(model.state_dict(), path)


def load(model: nn.Module, path: str | Path) -> nn.Module:
    model.load_state_dict(torch.load(path, weights_only=True))
    return model


# ---- Lesson 5: validation, early stopping, regularization ------------------

def fit_with_validation(model: nn.Module, X_train: Tensor, y_train: Tensor, X_val: Tensor, y_val: Tensor,
                        epochs: int = 300, lr: float = 0.01, optimizer_cls=torch.optim.Adam,
                        weight_decay: float = 0.0, patience: int | None = None, loss_fn: nn.Module | None = None) -> dict:
    """Train while watching the validation loss.

    Each epoch: one ``train_one_epoch`` on the training data, then ``evaluate``
    on the validation data. Keep a copy of the best state_dict seen so far
    (``copy.deepcopy(model.state_dict())`` when the validation loss improves).
    If ``patience`` is set, stop when the validation loss has not improved for
    ``patience`` epochs in a row. Before returning, load the best weights back
    into the model. Pass ``weight_decay`` to the optimizer.

    Returns {"train": [...], "val": [...], "best_epoch": int, "best_val": float}.
    """
    raise NotImplementedError("Lesson 5, step 3: the loop above; import copy for deepcopy")

In [ ]:
from forkcast import train
importlib.reload(train)
from forkcast.model import MLP

torch.manual_seed(0)
model = MLP(4, (32, 32))
h = train.fit_with_validation(model, Xn_tr, y_tr, Xn_val, y_val, epochs=300, lr=0.01)
print("baseline on 1400 deliveries: train", round(h["train"][-1], 2), "val", round(h["val"][-1], 2))

## 4. Overfit on purpose

40 deliveries, 4545 parameters, 800 epochs.

In [ ]:
small = Normalizer().fit(X_tr[:40])
Xs, ys, Xs_val = small.transform(X_tr[:40]), y_tr[:40], small.transform(X_val)
torch.manual_seed(0)
h = train.fit_with_validation(MLP(4, (64, 64)), Xs, ys, Xs_val, y_val, epochs=800, lr=0.01)
print("train", round(h["train"][-1], 2), "val", round(h["val"][-1], 2), "best val", round(h["best_val"], 2), "at epoch", h["best_epoch"])
plt.plot(h["train"], label="train"); plt.plot(h["val"], label="validation")
plt.axvline(h["best_epoch"], ls="--", c="k"); plt.yscale("log"); plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend();

## 5. Stop early, then regularize

Add `dropout` to `MLP` in the cell below: when `dropout > 0`, append `nn.Dropout(dropout)` after each activation.

In [ ]:
%%writefile forkcast/model.py
"""Lesson 3. The delivery-time model as an nn.Module (reference solution)."""

from __future__ import annotations

import torch
from torch import Tensor, nn


class LinearRegressor(nn.Module):
    """minutes = X @ w + b, with the knobs owned by an nn.Linear layer."""

    def __init__(self, n_features: int):
        super().__init__()                        # registers parameters, enables .to(), state_dict, ...
        self.linear = nn.Linear(n_features, 1)    # weight (1, d), bias (1,)

    def forward(self, X: Tensor) -> Tensor:
        """(N, d) -> (N,)"""
        return self.linear(X).squeeze(-1)         # (N, 1) -> (N,), so it lines up with y

# ---- Lesson 4: the first neural network -------------------------------------

def count_parameters(model: nn.Module) -> int:
    """Number of trainable numbers in ``model``."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


class MLP(nn.Module):
    """A multilayer perceptron: Linear -> activation [-> Dropout] -> ... -> Linear(…, 1).

    ``dropout`` (Lesson 5) is the fraction of hidden units zeroed on each training step; 0 adds no Dropout layers.
    """

    def __init__(self, n_features: int, hidden: tuple[int, ...] = (32, 32), activation=nn.ReLU, dropout: float = 0.0):
        super().__init__()
        layers: list[nn.Module] = []
        width = n_features
        for h in hidden:
            layers.append(nn.Linear(width, h))        # (N, width) -> (N, h)
            layers.append(activation())               # bends; without it two Linears are one Linear
            # Lesson 5, step 5: when dropout > 0, append nn.Dropout(dropout) here, after the activation
            width = h
        layers.append(nn.Linear(width, 1))            # the output layer has no activation
        self.net = nn.Sequential(*layers)

    def forward(self, X: Tensor) -> Tensor:
        """(N, d) -> (N,)"""
        return self.net(X).squeeze(-1)

In [ ]:
from forkcast import model as model_module
importlib.reload(model_module)
from forkcast.model import MLP
print(MLP(4, (64, 64), dropout=0.2).net)

torch.manual_seed(0)
m = MLP(4, (64, 64))
h = train.fit_with_validation(m, Xs, ys, Xs_val, y_val, epochs=800, lr=0.01, patience=30)
print("early stopping: ran", len(h["val"]), "epochs, best", round(h["best_val"], 2), "at", h["best_epoch"],
      "| model carries best weights:", round(train.evaluate(m, Xs_val, y_val, nn.MSELoss()), 2))

torch.manual_seed(0)
h = train.fit_with_validation(MLP(4, (64, 64), dropout=0.2), Xs, ys, Xs_val, y_val, epochs=800, lr=0.01, weight_decay=1e-2)
print("dropout 0.2 + weight decay 1e-2: best val", round(h["best_val"], 2), "at", h["best_epoch"],
      "| training loss with dropout on:", round(h["train"][-1], 1))

## 6. The learning curve

In [ ]:
for n in [50, 100, 200, 400, 800, 1400]:
    nm = Normalizer().fit(X_tr[:n]); torch.manual_seed(0)
    h = train.fit_with_validation(MLP(4, (32, 32)), nm.transform(X_tr[:n]), y_tr[:n], nm.transform(X_val), y_val,
                                  epochs=300, lr=0.01, patience=30)
    print(f"{n:5d} training deliveries -> best validation loss {h['best_val']:.2f}")

### Break it on purpose: cut without shuffling

In [ ]:
order = X[:, 0].argsort()                            # deliveries sorted by distance
Xs_, ys_ = X[order], y[order]
X_a, y_a, X_b, y_b = Xs_[:1400], ys_[:1400], Xs_[1700:], ys_[1700:]
nm = Normalizer().fit(X_a); torch.manual_seed(0); m = MLP(4, (32, 32))
train.fit_with_validation(m, nm.transform(X_a), y_a, nm.transform(Xs_[1400:1700]), ys_[1400:1700], epochs=300, lr=0.01)
print("test loss on the farthest 15%:", round(train.evaluate(m, nm.transform(X_b), y_b, nn.MSELoss()), 2),
      "| those deliveries start at", round(X_b[:, 0].min().item(), 1), "km")

## 7. The test loss, once

In [ ]:
print(round(train.evaluate(model, Xn_te, y_te, nn.MSELoss()), 2))

## 8. Definition of done

Expected: `7 passed`, then `41 passed` with no failures for everything so far (skipped tests belong to lessons you have not started).

In [ ]:
!pytest tests/test_05.py -q

In [ ]:
!pytest -m fast -q

## 9. Commit and push

In [ ]:
from getpass import getpass
TOKEN = getpass("GitHub token with access to your repo: ")

!git config user.name "{GITHUB_USER}"
!git config user.email "{GITHUB_USER}@users.noreply.github.com"
!git add -A
!git commit -m "Lesson 5: generalization"
!git push https://{TOKEN}@github.com/{GITHUB_USER}/{REPO}.git HEAD:main